# BrainScan AI

The implementation guide, step by step. Run the cells in order.

## Step 0: Setup and planning

Install the packages in `requirements.txt` and the Graphviz system package. Put one folder per class in `data/raw/`. Plan the Jira epics listed in the README.

## Step 1: Imports and config

Import the libraries and set the random seeds, image size, and dataset path.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import os, time, random, json
import cv2, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

SEED, IMG_SIZE = 42, 224
DATA_DIR = "data/raw"
VALID_EXT = {".jpeg", ".jpg", ".bmp", ".png"}
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


## Step 2: Check extensions and remove invalid files

Remove unsupported extensions. Back up the original dataset first because deletion is permanent.

In [ ]:
def clean_dataset(root):
    removed = 0
    for cls in os.listdir(root):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if os.path.splitext(f)[1].lower() not in VALID_EXT:
                os.remove(os.path.join(cls_dir, f))
                removed += 1
    print(f"Removed {removed} invalid files")

# After backing up your data, uncomment this:
# clean_dataset(DATA_DIR)


## Step 3: Load, resize, shuffle (with try/except)

OpenCV reads BGR images. Convert to RGB, resize, and shuffle images and labels together. Keep images as uint8 until normalization.

In [ ]:
def load_dataset(root, img_size=IMG_SIZE):
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    print("Classes:", classes)
    images, labels = [], []
    for cls in classes:
        cls_dir = os.path.join(root, cls)
        for f in os.listdir(cls_dir):
            path = os.path.join(cls_dir, f)
            try:
                img = cv2.imread(path)
                if img is None:
                    raise ValueError("unreadable or corrupted image")
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
                images.append(img)
                labels.append(cls)
            except Exception as e:
                print(f"Skipped {path}: {e}")

    # Shuffle both lists together so index i still matches
    combined = list(zip(images, labels))
    random.shuffle(combined)
    images, labels = zip(*combined)
    return np.array(images, dtype=np.uint8), np.array(labels), classes

X, y, classes = load_dataset(DATA_DIR)
print(X.shape, y.shape)   # (N, 224, 224, 3), (N,)


## Step 4: Explore the data

Inspect class counts and five images from each class.

In [ ]:
# Class distribution
unique, counts = np.unique(y, return_counts=True)
plt.figure(figsize=(7,4))
sns.barplot(x=unique, y=counts)
plt.title("Images per class"); plt.ylabel("Count")
plt.show()

# 5 samples per class
fig, axes = plt.subplots(len(classes), 5, figsize=(12, 3*len(classes)))
for i, cls in enumerate(classes):
    idx = np.where(y == cls)[0][:5]
    for j, k in enumerate(idx):
        axes[i, j].imshow(X[k]); axes[i, j].axis("off")
        if j == 0: axes[i, j].set_title(cls, loc="left")
plt.show()


## Step 5: Encode labels and split

Encode labels, save their order, and use a stratified 80/20 train/test split.

In [ ]:
le = LabelEncoder()
y_int = le.fit_transform(y)
y_cat = to_categorical(y_int)
os.makedirs("models", exist_ok=True)
json.dump(list(le.classes_), open("models/classes.json", "w"))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, stratify=y_int, random_state=SEED)


## Step 6: Check balance and rebalance if needed

Compute class weights from training labels only. Do not rebalance before splitting.

In [ ]:
weights = compute_class_weight("balanced", classes=np.unique(y_int),
                               y=np.argmax(y_train, axis=1))
class_weights = dict(enumerate(weights))


## Step 7: Normalize to [0, 1]

Convert pixels to float32 and divide by 255 to get values in [0, 1]. This step needs several GB of RAM for the full dataset.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0


## Step 8: Build the CNN

Use four convolution/pooling blocks, dropout, global average pooling, and a softmax classifier. Compile with Adam and categorical cross-entropy.

In [ ]:
def build_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=len(classes)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()
os.makedirs("reports/figures", exist_ok=True)
plot_model(model, to_file="reports/figures/model.png", show_shapes=True)


## Step 9: Train with timing and checkpoint

Train for up to 30 epochs with batch size 32. Save the best validation-accuracy checkpoint and monitor validation loss for early stopping and learning-rate reduction.

In [ ]:
callbacks = [
    ModelCheckpoint("models/best_model.h5", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

start = time.time()
history = model.fit(X_train, y_train, validation_split=0.1,
                    epochs=30, batch_size=32,
                    class_weight=class_weights, callbacks=callbacks)
print(f"Training time: {(time.time()-start)/60:.1f} min")


## Step 10: Evaluate

Inspect accuracy, per-class recall, learning curves, the confusion matrix, and correct versus incorrect predictions.

In [ ]:
# Evaluate the same checkpoint used by the app.
model = tf.keras.models.load_model("models/best_model.h5")

# Test performance
loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")

# Learning curves
fig, ax = plt.subplots(1, 2, figsize=(12,4))
for a, m in zip(ax, ["accuracy", "loss"]):
    a.plot(history.history[m], label="train")
    a.plot(history.history[f"val_{m}"], label="val")
    a.set_title(m); a.legend()
plt.show()

# Confusion matrix + report
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Correct vs incorrect examples
wrong = np.where(y_pred != y_true)[0][:6]
right = np.where(y_pred == y_true)[0][:6]
for title, idxs in [("Correct", right), ("Incorrect", wrong)]:
    plt.figure(figsize=(14,3))
    for i, k in enumerate(idxs):
        plt.subplot(1, 6, i+1); plt.imshow(X_test[k]); plt.axis("off")
        plt.title(f"T:{le.classes_[y_true[k]]}\nP:{le.classes_[y_pred[k]]}", fontsize=8)
    plt.suptitle(title); plt.show()


## Step 11: Deployment

Use the saved checkpoint and class order with the same RGB conversion, resizing, and normalization. Run `streamlit run app/streamlit_app.py` from the project folder to use the upload interface.

In [ ]:
import json, cv2, numpy as np
from tensorflow.keras.models import load_model

model = load_model("models/best_model.h5")
classes = json.load(open("models/classes.json"))

def predict_image(path, img_size=224):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Cannot read image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size)).astype("float32") / 255.0
    probs = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    k = int(np.argmax(probs))
    return classes[k], float(probs[k])


## Step 12: Documentation, reproducibility, Jira

Record actual test accuracy and per-class recall after training. The README contains setup instructions, project structure, limitations, and the six Jira epics. Use one-week sprints. Watch for duplicate images and multiple slices from the same patient across partitions. This is decision support, not a medical diagnosis.